# <font style="font-family:roboto;color:#455e6c"> Calculation of phase diagrams </font>  

<div class="admonition note" name="html-admonition" style="background:#e3f2fd; padding: 10px">
<font style="font-family:roboto;color:#455e6c"> <b> DPG Tutorial: Automated Workflows and Machine Learning for Materials Science Simulations </b> </font> </br>
<font style="font-family:roboto;color:#455e6c"> 8 March 2026 </font> </br> </br>
Sriram Anand, Prabhath Chilakalapudi, Haitham Gaafer, Marvin Poul, Jan Janssen, Jörg Neugebauer </br>
<i> Max Planck Institute for Sustainable Materials </i></br>
</br>
Sarath Menon, Minaam Qamar, Ralf Drautz </br>
<i> Ruhr-Universität Bochum </i></br>
</br>
Tilmann Hickel </br>
<i> Bundesanstalt für Materialforschung und -prüfung </i></br>
</div>

## <font style="font-family:roboto;color:#455e6c"> Calculate free energy with temperature for solid</font> 

In [ ]:
from core import Workflow
from core.gui import PyironFlow

from pyiron_nodes.atomistic.structure.build import Bulk
from pyiron_nodes.atomistic.structure.transform import Repeat
from pyiron_nodes.atomistic.calculator.calphy import (
    InputClass,
    SolidFreeEnergyWithTemp,
    PlotFreeEnergy
)

from pyiron_nodes.atomistic.engine.lammps import ListPotentials
from pyiron_nodes.atomistic.engine.eam import EAM
from pyiron_nodes.basic.list import PickElement

In [ ]:
wf_free_energy_unary = Workflow("Free_Energy_Unary")
wf_free_energy_unary_solution = Workflow("Free_Energy_Unary_Solution")

In [ ]:
wf_free_energy_unary_solution.BulkStructure = Bulk(name="Al", cubic=True)
wf_free_energy_unary_solution.RepeatStructure = Repeat(structure=wf_free_energy_unary_solution.BulkStructure, repeat_scalar=3)
wf_free_energy_unary_solution.ListPotentials = ListPotentials(structure=wf_free_energy_unary_solution.BulkStructure)
wf_free_energy_unary_solution.PickElement = PickElement(lst=wf_free_energy_unary_solution.ListPotentials.outputs.potentials, index=20)
wf_free_energy_unary_solution.EAM = EAM(potential_name=wf_free_energy_unary_solution.PickElement.outputs.element)

wf_free_energy_unary_solution.CalphyInputClass = InputClass()
wf_free_energy_unary_solution.SolidFreeEnergyWithTemp = SolidFreeEnergyWithTemp(
    input_class=wf_free_energy_unary_solution.CalphyInputClass,
    structure=wf_free_energy_unary_solution.RepeatStructure,
    potential=wf_free_energy_unary_solution.EAM.outputs.engine,
    delete_folder=False,
)
wf_free_energy_unary_solution.PlotUnaryFreeEnergy = PlotFreeEnergy(
    temperature=wf_free_energy_unary_solution.SolidFreeEnergyWithTemp.outputs.temperature, 
    free_energy=wf_free_energy_unary_solution.SolidFreeEnergyWithTemp.outputs.free_energy
)

In [ ]:
workflow_path = "pyiron_core_data/workflows"
pf_free_energy_unary = PyironFlow(
    wf_list=[wf_free_energy_unary, wf_free_energy_unary_solution],
    nodes_path="pyiron_nodes", 
    workflow_path=workflow_path
)
pf_free_energy_unary.gui

## <font style="font-family:roboto;color:#455e6c"> Calculate melting temperature </font> 

In [ ]:
from core import Workflow
from core.gui import PyironFlow

from pyiron_nodes.atomistic.structure.build import Bulk
from pyiron_nodes.atomistic.structure.transform import Repeat, Rattle
from pyiron_nodes.atomistic.calculator.calphy import (
    InputClass,
    SolidFreeEnergyWithTemp,
    LiquidFreeEnergyWithTemp
)
from pyiron_nodes.thermodynamics.landau import (
    TemperatureDependentLinePhase,
    TransitionTemperature
)

from pyiron_nodes.atomistic.engine.lammps import ListPotentials
from pyiron_nodes.atomistic.engine.eam import EAM
from pyiron_nodes.basic.list import PickElement

In [ ]:
wf_transition_temp_unary = Workflow("Transition_Temp_Unary")
wf_transition_temp_unary_solution = Workflow("Transition_Temp_Unary_Solution")

In [ ]:
wf_transition_temp_unary_solution.BulkStructure = Bulk(name="Al", cubic=True)
wf_transition_temp_unary_solution.RepeatStructure = Repeat(structure=wf_transition_temp_unary_solution.BulkStructure, repeat_scalar=3)
wf_transition_temp_unary_solution.RattleStructure = Rattle(structure=wf_transition_temp_unary_solution.RepeatStructure, seed=42, stdev=0.6)

wf_transition_temp_unary_solution.ListPotentials = ListPotentials(structure=wf_transition_temp_unary_solution.BulkStructure)
wf_transition_temp_unary_solution.PickElement = PickElement(lst=wf_transition_temp_unary_solution.ListPotentials.outputs.potentials, index=20)
wf_transition_temp_unary_solution.EAM = EAM(potential_name=wf_transition_temp_unary_solution.PickElement.outputs.element)

wf_transition_temp_unary_solution.CalphyInputClass = InputClass(temperature=900, temperature_stop=1100)

wf_transition_temp_unary_solution.SolidFreeEnergyWithTemp = SolidFreeEnergyWithTemp(
    input_class=wf_transition_temp_unary_solution.CalphyInputClass,
    structure=wf_transition_temp_unary_solution.RepeatStructure, 
    potential=wf_transition_temp_unary_solution.EAM.outputs.engine,
)
wf_transition_temp_unary_solution.LiquidFreeEnergyWithTemp = LiquidFreeEnergyWithTemp(
    input_class=wf_transition_temp_unary_solution.CalphyInputClass,
    structure=wf_transition_temp_unary_solution.RattleStructure,
    potential=wf_transition_temp_unary_solution.EAM.outputs.engine,
)

wf_transition_temp_unary_solution.SolidLinePhase = TemperatureDependentLinePhase(
    name="AlSolid", 
    fixed_concentration=0,
    temperatures=wf_transition_temp_unary_solution.SolidFreeEnergyWithTemp.outputs.temperature, 
    free_energies=wf_transition_temp_unary_solution.SolidFreeEnergyWithTemp.outputs.free_energy
)
wf_transition_temp_unary_solution.LiquidLinePhase = TemperatureDependentLinePhase(
    name="AlLiquid",
    fixed_concentration=0,
    temperatures=wf_transition_temp_unary_solution.LiquidFreeEnergyWithTemp.outputs.temperature,
    free_energies=wf_transition_temp_unary_solution.LiquidFreeEnergyWithTemp.outputs.free_energy
)
wf_transition_temp_unary_solution.TransitionTemperature = TransitionTemperature(
    phase1=wf_transition_temp_unary_solution.SolidLinePhase, 
    phase2=wf_transition_temp_unary_solution.LiquidLinePhase, 
    Tmin=800, 
    Tmax=1100
)

In [ ]:
workflow_path = "pyiron_core_data/workflows"
pf_transition_temp_unary = PyironFlow(
    wf_list=[wf_transition_temp_unary, wf_transition_temp_unary_solution], 
    nodes_path="pyiron_nodes", 
    workflow_path=workflow_path
)

pf_transition_temp_unary.gui

## <font style="font-family:roboto;color:#455e6c"> Calculating a binary phase diagram </font> 

Now that we have seen the thermodynamic functions in Landau, we will make a simple eutectic binary phase diagram with Landau.

In [ ]:
from core import Workflow
from core.gui import PyironFlow

from pyiron_nodes.thermodynamics.landau import (
    LinePhase, 
    IdealSolution, 
    TemperatureDependentLinePhase, 
    CalcPhaseDiagram, 
    PlotConcPhaseDiagram
)
from pyiron_nodes.basic.math import Linspace
from pyiron_nodes.basic.list import List5


In [ ]:
wf_phase_diagram_binary = Workflow("Phase_Diagram_Binary_Solution")

In [ ]:
wf_phase_diagram_binary.SolidLinePhaseA = LinePhase(name="SolidA", fixed_concentration=0, line_energy=2.45, line_entropy=0.00001)
wf_phase_diagram_binary.SolidLinePhaseB = LinePhase(name="SolidB", fixed_concentration=1, line_energy=2.95, line_entropy=0.00001)
wf_phase_diagram_binary.LiquidLinePhaseA = LinePhase(name="LiquidA", fixed_concentration=0, line_energy=2.5, line_entropy=0.0001)
wf_phase_diagram_binary.LiquidLinePhaseB = LinePhase(name="LiquidB", fixed_concentration=1, line_energy=3.1, line_entropy=0.0002)
wf_phase_diagram_binary.IdealSolution = IdealSolution(
    name="liquid", 
    phase1=wf_phase_diagram_binary.LiquidLinePhaseA, 
    phase2=wf_phase_diagram_binary.LiquidLinePhaseB
)
wf_phase_diagram_binary.TemperaturesLinspace = Linspace(
    x_min=250,
    x_max=1000,
    num_points=25
)
wf_phase_diagram_binary.PhasesList = List5(
    x1=wf_phase_diagram_binary.SolidLinePhaseA,
    x2=wf_phase_diagram_binary.SolidLinePhaseB,
    x3=wf_phase_diagram_binary.IdealSolution
)
wf_phase_diagram_binary.CalcPhaseDiagram = CalcPhaseDiagram(
    phases=wf_phase_diagram_binary.PhasesList,
    temperatures=wf_phase_diagram_binary.TemperaturesLinspace,
    chemical_potentials=50
)
wf_phase_diagram_binary.PlotConcPhaseDiagram = PlotConcPhaseDiagram(
    phase_data=wf_phase_diagram_binary.CalcPhaseDiagram,
    plot_tielines=True
)

In [ ]:
workflow_path = "pyiron_core_data/workflows"
pf_phase_diagram_binary = PyironFlow(
    wf_list=[wf_phase_diagram_binary], 
    nodes_path="pyiron_nodes", 
    workflow_path=workflow_path
)

pf_phase_diagram_binary.gui

## <font style="font-family:roboto;color:#455e6c"> MgCa phase diagram </font> 

Now we will take the same approach to calculate a phase diagram with real data, in this case the MgCa system. The data can be found [here](https://github.com/eisenforschung/mgalca-mtp-data).

In [ ]:
from core import Workflow
from core.gui import PyironFlow

from pyiron_nodes.basic.file import ReadDataFrame
from pyiron_nodes.basic.math import Linspace
from pyiron_nodes.thermodynamics.landau import (
    PhasesFromDataFrame, 
    CalcPhaseDiagram, 
    PlotConcPhaseDiagram, 
    PlotMuPhaseDiagram
)

In [ ]:
wf_phase_diagram_MgCa = Workflow("Phase_Diagram_MgCa_Solution")

In [ ]:
wf_phase_diagram_MgCa.ReadMgCaDataFrame = ReadDataFrame(filename="data/MgCaFreeEnergies.pckl.gz", compression="gzip")
wf_phase_diagram_MgCa.PhasesFromDataFrame = PhasesFromDataFrame(dataframe=wf_phase_diagram_MgCa.ReadMgCaDataFrame)
wf_phase_diagram_MgCa.TemperaturesLinspace = Linspace(x_min=200, x_max=1200, num_points=50)
wf_phase_diagram_MgCa.CalculatePhaseDiagram = CalcPhaseDiagram(
    phases=wf_phase_diagram_MgCa.PhasesFromDataFrame.outputs.phase_list,
    temperatures=wf_phase_diagram_MgCa.TemperaturesLinspace,
    chemical_potentials=50
)
wf_phase_diagram_MgCa.PlotConcentrationPhaseDiagram = PlotConcPhaseDiagram(phase_data=wf_phase_diagram_MgCa.CalculatePhaseDiagram, plot_tielines=True)
wf_phase_diagram_MgCa.PlotMuPhaseDiagram = PlotMuPhaseDiagram(phase_data=wf_phase_diagram_MgCa.CalculatePhaseDiagram)

In [ ]:
workflow_path = "pyiron_core_data/workflows"
pf_phase_diagram_MgCa = PyironFlow(
    wf_list=[wf_phase_diagram_MgCa], 
    nodes_path="pyiron_nodes", 
    workflow_path=workflow_path
)

pf_phase_diagram_MgCa.gui

---
---

### <font style="font-family:roboto;color:#455e6c"> Software used in this notebook </font>  

- [calphy](https://calphy.org)
- [landau](https://github.com/eisenforschung/landau)
- [LAMMPS](https://www.lammps.org/)
- [pyiron](https://pyiron.org/)
- [pyiron_workflow](https://github.com/pyiron/pyiron_workflow)

<img src="img/logo_roll.png" width="1200">